# 🚀 Zynthe Cloud Backend
Run this notebook to start the heavy distillation backend on Google's free GPUs.

**Steps:**
1.  **Runtime Check:** Ensure you are connected to a GPU (Runtime -> Change runtime type -> T4 GPU).
2.  **Config:** Enter your ngrok token below.
3.  **Run All:** Click `Runtime -> Run all` (or Ctrl+F9).

In [ ]:
# @title 1. Environment Setup & GPU Check
import torch
import sys

if not torch.cuda.is_available():
    print("⚠️  WARNING: GPU not detected!")
    print("Please go to 'Runtime -> Change runtime type' and select 'T4 GPU'.")
    # Uncomment to enforce failure
    # raise RuntimeError("No GPU detected. Please enable T4 GPU in Runtime settings.")
else:
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

print("📦 Installing dependencies... (this takes ~2 mins)")
!git clone https://github.com/lakshin7/zynthe.git
%cd zynthe
!pip install -q -r ui/backend/requirements.txt
!pip install -q pyngrok uvicorn fastapi nesting-asyncio
print("✅ Environment Ready")

In [ ]:
# @title 2. Start Backend & Tunnel
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import os

# ---------------------------------------------------------
NGROK_TOKEN = "" # @param {type:"string"}
# ---------------------------------------------------------

if not NGROK_TOKEN:
    print("❌ Error: Please enter your ngrok token above.")
    print("Get it for free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    # Apply patch for Colab's event loop
    nest_asyncio.apply()

    # Set token
    ngrok.set_auth_token(NGROK_TOKEN)

    # Close existing tunnels
    ngrok.kill()

    # Open a tunnel to port 8765
    try:
        public_url = ngrok.connect(8765).public_url
        print("="*60)
        print("🚀 BACKEND IS LIVE!")
        print(f"🔗 Copy this URL to your Local App: {public_url.replace('http://', 'wss://').replace('https://', 'wss://')}/ws")
        print("="*60)

        # Add path for imports
        sys.path.append(os.getcwd())
        
        from ui.backend.api import app
        
        # Run the server
        uvicorn.run(app, port=8765)
    except Exception as e:
        print(f"❌ Failed to start: {e}")